# Notebook 4: Segmentation & Policy Scenario Analysis
## Loan Default Risk Analysis | RISK-2024-001
**Author:** Gnanesh (BA)

**Purpose:**
1. Identify the top high-risk borrower segments
2. Quantify each segment's contribution to total losses
3. Model the impact of the recommended policy changes

---
**Run notebooks 02 and 03 first.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

df = pd.read_csv('../data/clean/loans_clean.csv', low_memory=False)
PORTFOLIO_AVG = df['default_flag'].mean() * 100
print(f'Rows: {len(df):,} | Default rate: {PORTFOLIO_AVG:.2f}%')

## 1. High-Risk Segment Identification

In [ ]:
# Identify segments with default rate ≥ 2x portfolio average
# Segment = grade + dti_band combination

seg = df.groupby(['grade', 'dti_band']).agg(
    loan_count=('default_flag', 'count'),
    defaults=('default_flag', 'sum'),
    total_loan_amnt=('loan_amnt', 'sum')
).reset_index()

seg['default_rate'] = seg['defaults'] / seg['loan_count'] * 100
seg['pct_of_total_defaults'] = seg['defaults'] / seg['defaults'].sum() * 100
seg['pct_of_volume'] = seg['loan_count'] / seg['loan_count'].sum() * 100

# Filter: at least 200 loans, and default rate ≥ 2x average
high_risk = seg[
    (seg['loan_count'] >= 200) &
    (seg['default_rate'] >= 2 * PORTFOLIO_AVG)
].sort_values('pct_of_total_defaults', ascending=False)

print(f'High-risk segments identified (≥2x avg, ≥200 loans): {len(high_risk)}')
high_risk[['grade', 'dti_band', 'loan_count', 'default_rate', 'pct_of_total_defaults', 'pct_of_volume']].head(10)

In [ ]:
# Total contribution of high-risk segments
total_defaults = df['default_flag'].sum()
total_loans = len(df)

hr_defaults = high_risk['defaults'].sum()
hr_loans = high_risk['loan_count'].sum()

print(f'High-risk segments: {hr_loans:,} loans ({hr_loans/total_loans*100:.1f}% of volume)')
print(f'High-risk segment defaults: {hr_defaults:,} ({hr_defaults/total_defaults*100:.1f}% of all defaults)')

## 2. Segment Contribution Chart

In [ ]:
# Top 10 grade × dti segments by % of total defaults
top_seg = seg.nlargest(10, 'pct_of_total_defaults').copy()
top_seg['label'] = top_seg['grade'] + ' / DTI ' + top_seg['dti_band'].astype(str)

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(top_seg['label'], top_seg['pct_of_total_defaults'],
              color='#d62728', edgecolor='white', alpha=0.85)
ax.set_title('Top 10 Segments by % Contribution to Total Defaults', fontsize=13)
ax.set_ylabel('% of Total Defaults')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

print('\n📌 INSIGHT: A small number of Grade D–G / High-DTI segments drive a large share of defaults.')

## 3. Scenario Analysis: Impact of Policy Changes

In [ ]:
def policy_scenario(df, label, grade_exclude=None, max_dti=None):
    """Compute default rate and volume impact of a policy change."""
    filtered = df.copy()
    if grade_exclude:
        filtered = filtered[~filtered['grade'].isin(grade_exclude)]
    if max_dti is not None:
        filtered = filtered[filtered['dti'] <= max_dti]
    
    return {
        'Scenario': label,
        'Loans': len(filtered),
        'Volume %': len(filtered) / len(df) * 100,
        'Defaults': filtered['default_flag'].sum(),
        'Default Rate %': filtered['default_flag'].mean() * 100,
        'Volume Reduction %': (1 - len(filtered)/len(df)) * 100
    }

scenarios = [
    policy_scenario(df, 'Current policy (baseline)'),
    policy_scenario(df, 'Option A: DTI ≤ 35%', max_dti=35),
    policy_scenario(df, 'Option B: Suspend Grade F & G', grade_exclude=['F','G']),
    policy_scenario(df, 'Option C: DTI ≤ 35% + Suspend F & G', grade_exclude=['F','G'], max_dti=35),
]

results = pd.DataFrame(scenarios)
results = results.round(2)
print(results.to_string(index=False))

In [ ]:
# Visualise scenario comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

# Default rate comparison
ax1.barh(results['Scenario'], results['Default Rate %'], color=colors, edgecolor='white')
ax1.set_xlabel('Default Rate (%)')
ax1.set_title('Default Rate by Scenario', fontsize=12)
ax1.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(ax1.patches, results['Default Rate %']):
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}%', va='center', fontsize=10)

# Volume reduction comparison
ax2.barh(results['Scenario'], results['Volume Reduction %'], color=colors, edgecolor='white')
ax2.set_xlabel('Volume Reduction (%)')
ax2.set_title('Approval Volume Reduction by Scenario', fontsize=12)
ax2.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(ax2.patches, results['Volume Reduction %']):
    ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=10)

plt.suptitle('Scenario Analysis: Default Rate vs Volume Trade-off', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\n📌 INSIGHT: Option C achieves the biggest default rate reduction with the largest volume trade-off.')
print('The phased approach (A in month 1, B in month 2) keeps volume drop manageable.')

## 4. Expected Loss Estimation (Simplified)

In [ ]:
# EL = PD × LGD × EAD
# Assumption: LGD = 60% (typical unsecured personal loan recovery rate of 40%)
LGD = 0.60

for _, row in results.iterrows():
    pd_rate = row['Default Rate %'] / 100
    # Approximate EAD as average loan amount (using overall avg as proxy)
    ead = df['loan_amnt'].mean()
    el_per_loan = pd_rate * LGD * ead
    print(f"{row['Scenario']:50s} | EL per loan: ${el_per_loan:,.0f} | Projected EL (portfolio): ${el_per_loan * row['Loans']:,.0f}")

## 5. Final Segment Summary Table

In [ ]:
# Summary table for inclusion in the Recommendation Memo
grade_summary = df.groupby('grade').agg(
    loan_count=('default_flag', 'count'),
    defaults=('default_flag', 'sum')
).reset_index()
grade_summary['default_rate'] = (grade_summary['defaults'] / grade_summary['loan_count'] * 100).round(1)
grade_summary['pct_portfolio'] = (grade_summary['loan_count'] / grade_summary['loan_count'].sum() * 100).round(1)
grade_summary['pct_defaults'] = (grade_summary['defaults'] / grade_summary['defaults'].sum() * 100).round(1)

print('Grade-level summary (for Recommendation Memo table):')
print(grade_summary.to_string(index=False))

## 6. Key Outputs for Recommendation Memo

Copy these numbers into `docs/08_recommendation_memo.md`:

| Policy Change | Default Rate | Volume Reduction |
|--------------|-------------|------------------|
| Baseline | See cell 3 output | 0% |
| DTI ≤ 35% | See cell 3 output | See cell 3 output |
| Suspend F & G | See cell 3 output | See cell 3 output |
| Combined | See cell 3 output | See cell 3 output |

> **Run all cells and replace the numbers above with the actual outputs before submitting the project.**